# Day 1 — Trade Ticket Email Parser

**Pattern:** Pydantic structured outputs from an LLM  
**Domain:** Capital Markets  
**Course reference:** Lab 1

---

## Step 1 — Setup

Install the packages and configure your two LLM providers. Never hardcode keys — use python-dotenv with a .env file that is in your .gitignore.

In [ ]:
# Install required packages and provide SDKs
!pip install openai anthropic pydantic python-dotenv

from dotenv import load_dotenv
from pathlib import Path
import os
import time

# Load all provider keys from a local .env file if present
dotenv_path = Path('.') / '.env'
load_dotenv(dotenv_path=dotenv_path)

# Verify all keys are loaded 
for key in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
    status = "✅ Loaded" if os.environ.get(key) else "❌ MISSING"
    print(f"{key}:{status}")

# Instantiate both clients — no API keys in code
from openai import OpenAI
from anthropic import Anthropic

# Create client instances using env vars (clients handle auth internally)
openai_client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
anthropic_client = Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))

## Step 2 — Define the Pydantic model

A trade ticket has: side (BUY or SELL), quantity (positive integer), symbol (uppercase ticker), order type (MARKET or LIMIT), limit price (optional, required if order_type is LIMIT), fund identifier, and settlement convention (T+0/T+1/T+2). Build this as a Pydantic BaseModel with appropriate types and validators.

In [ ]:
# import pydantic and typing for the TradeTicket model
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Literal, Optional

# define a TradeTicket model with the following fields and validations:
class TradeTicket(BaseModel):
   side: Literal['BUY', 'SELL']
   quantity: int = Field(gt=0)        # positive only
   symbol: str                         # validate uppercase
   order_type: Literal['MARKET', 'LIMIT']
   limit_price: Optional[float] = None
   fund_id: str
   settlement: Literal['T+0', 'T+1', 'T+2']

# Add a @field_validator for symbol that uppercases the value
   @field_validator('symbol')
   def uppercase_symbol(cls, v):
       return v.upper() if isinstance(v, str) else v

# Add a @model_validator that enforces: if order_type='LIMIT' then limit_price required
   @model_validator(mode = 'after')
   def validate_model(self):
       if self.order_type == 'LIMIT' and self.limit_price is None:
           raise ValueError('limit_price required for LIMIT orders')
       return self


## Step 3 — Call the LLM with structured outputs

Both OpenAI and Anthropic support returning JSON that conforms to a Pydantic schema. OpenAI calls this 'structured outputs' (response_format with json_schema). Anthropic uses tool-use with an input_schema. Pick one for this lab and stick with it.

In [ ]:
# Call the LLM and return a parsed TradeTicket
import json
from pydantic import ValidationError

# System prompt used for all calls
system_prompt1 = "You extract trade tickets from emails. Return JSON matching the schema"

def call_openai(prompt, system_prompt= """You extract trade tickets from emails. Return ONLY raw JSON (no markdown, no explanation) with exactly these fields:
- side: "BUY" or "SELL"
- quantity: positive integer
- symbol: uppercase ticker
- order_type: "MARKET" or "LIMIT"
- limit_price: number or null
- fund_id: string
- settlement: "T+0", "T+1", or "T+2"
""", model="gpt-4.1-mini", temperature=0.0):
    """Call OpenAI Chat Completions API and return a simple dict with text and timing."""
    #start = time.time()
    response = openai_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    #latency = time.time() - start
    # Attempt to extract the assistant text from the response object/dict
    text = None
    try:
        text = response.choices[0].message.content
    except Exception:
        try:
            text = response['choices'][0]['message']['content']
        except Exception:
            text = str(response)
    return {"text": text, "model": model}#

def parse_trade_email(email_text: str) -> TradeTicket:
    """Use the LLM to parse `email_text` into a TradeTicket instance."""
    # Build the user prompt containing the email body
    prompt = f"Extract a JSON object matching the TradeTicket schema from this email body:\n\n{email_text}"
    resp = call_openai(prompt)
    raw = resp.get('text', '')
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1]       # remove first ```json line
        raw = raw.rsplit("```", 1)[0]     # remove closing ```
        raw = raw.strip()
    # Attempt to parse JSON from the model output
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"LLM returned non-JSON response: {e}\nRaw output:\n{raw}")
    # Validate into TradeTicket (Pydantic v2 style)
    try:
        ticket = TradeTicket.model_validate(parsed)
        return ticket
    except ValidationError as e:
        # Re-raise the validation error so the caller can add to review queue
        raise

# Write a small list of test emails — at least 5, including ONE deliberately ambiguous one
test_emails = [
    "Please buy 10000 MSFT at market for the Growth Fund, settlement T+1.",
    "Sell 5k AAPL limit 175.50 for fund GAF with T+1 settlement.",
    "BUY 250 VOO MARKET for Fund Alpha, settlement T+2.",
    "Sell 150 TSLA at limit 620.00 for fund ZETA, settle T+0.",
    "Buy some shares of Apple for the Growth Fund.",  # Deliberately ambiguous — missing quantity, no settlement specified
    "LONG 500 SPY @market for Tactical Fund T+2.",
    "Short 300 QQQ limit 400 fund Beta settlement T+1.",
]

# Run tests: collect failures into a human_review_queue
human_review_queue = []
try:
    for email in test_emails:
        try:
            ticket = parse_trade_email(email)
            print(f"✅ Parsed: {ticket}")
        except ValidationError as e:
            print(f"❌ Validation failed for email: {email}")
            print(f"Error: {e}")
            human_review_queue.append({"email": email, "error": str(e)})
        except Exception as e:
            print(f"⚠️ Parsing error for email: {email} -> {e}")
            human_review_queue.append({"email": email, "error": str(e)})
except Exception as e:
    print(f"Unexpected error: {e}")

# Print summary
print(f"\nTotal processed: {len(test_emails)}")
print(f"Successfully parsed: {len(test_emails) - len(human_review_queue)}")
print(f"Sent to manual review: {len(human_review_queue)}")

# If last parsed ticket exists, print it
try:
    print(ticket)
except NameError:
    pass


## Step 4 — Handle validation failures

If the LLM returns JSON that does not validate, you get a ValidationError. In production this is your most valuable signal — it is the system telling you 'this needs a human'. Catch the error, log the raw response, and write the email to a 'manual review' queue (just a list or file for this lab).

In [ ]:
from pydantic import ValidationError

# Reset queues for a clean run
parsed_tickets = []
human_review_queue = []

for email in test_emails:
    try:
        ticket = parse_trade_email(email)
        parsed_tickets.append(ticket)
        print(f"✅ Parsed: {ticket.model_dump_json(indent=2)}")

    except ValidationError as e:
        print(f"❌ Validation failed for: {email}")
        print(f"   Errors: {e.error_count()} field(s) rejected")
        for err in e.errors():
            print(f"   - {err['loc'][0]}: {err['msg']}")
        human_review_queue.append({"email": email, "error_type": "validation", "details": e.errors()})

    except ValueError as e:
        # This catches the JSON-parse failures from parse_trade_email
        print(f"⚠️ JSON parse failed for: {email}")
        print(f"   {e}")
        human_review_queue.append({"email": email, "error_type": "json_parse", "details": str(e)})

    except Exception as e:
        print(f"🔥 Unexpected error for: {email}")
        print(f"   {type(e).__name__}: {e}")
        human_review_queue.append({"email": email, "error_type": "unexpected", "details": str(e)})

# Summary
print(f"\n{'='*50}")
print(f"Total processed:      {len(test_emails)}")
print(f"Successfully parsed:  {len(parsed_tickets)}")
print(f"Sent to manual review: {len(human_review_queue)}")

# Show the review queue contents
if human_review_queue:
    print(f"\n--- Manual Review Queue ---")
    for i, item in enumerate(human_review_queue, 1):
        print(f"{i}. [{item['error_type']}] {item['email'][:60]}...")